# NC coherent 1γ reference files: TestSplit vs BatchB, and the 2.22e24 POT

`src/coh_1g_reweighting.py` reweights isotropic-1γ (iso1g) events to the NC coherent 1γ photon (cosθ, E) distribution and normalizes them with
`reference_nc_coherent_pot = 2.22242e24` (from Mark, slack 2026-02-24), `reference_nc_coherent_num_events = 16139` and `train_test_fudge_factor = 0.5`.

Two NC coherent reference files exist in `other_files_location`:

| file | origin |
|---|---|
| `preTopo_vertex_NCSingleCoherent_Run123_TestSplit_v50.0_OrderedSample_NCDelta.uniq.root` (**TestSplit**, used now) | `/exp/uboone/data/users/gge/SinglePhotonAnalysis/FluxEventweightAssignment/` |
| `vertex_NCSingleCoherent_Run123_BatchB_v50.0.uniq.root` (**BatchB**) | `/pnfs/uboone/persistent/users/markross/Jan2022_gLEE_files/UniqDir/` |

Questions:
1. How are the two files related, and is either a biased subset?
2. Which number of generated events does the 2.22242e24 POT correspond to, and does that match the paper (arXiv:2502.06091)?
3. What does this imply for the iso1g reweighting normalization?

Conclusions are summarized at the bottom.

In [ ]:
import os, sys
import numpy as np
import polars as pl
import uproot
import matplotlib.pyplot as plt
import matplotlib as mpl

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.file_locations import other_files_location, intermediate_files_location

TS_FILE = "preTopo_vertex_NCSingleCoherent_Run123_TestSplit_v50.0_OrderedSample_NCDelta.uniq.root"
BB_FILE = "vertex_NCSingleCoherent_Run123_BatchB_v50.0.uniq.root"

MARK_POT = 2.22242e24          # from Mark, slack 2026-02-24
PAPER_POT = 6.87e20            # arXiv:2502.06091

# Paper numbers (arXiv:2502.06091):
#  "This topological requirement selects 28.1% of simulated NC coherent 1γ events"
#  (relative to all simulated signal events in the TPC; "At the preselection stage, 27.7% ...
#   remains out of all simulated signal events in the TPC")
#  Table 2: 1.3 signal events in the single-photon selection, 1.1 in the coherent-rich selection
#  Table 3 (relative to topological): 45.85% single-photon, 39.09% coherent-rich
paper_eff_topo = 0.281
paper_eff_rel_1g = 0.4585
paper_eff_rel_coh = 0.3909
paper_n_1g = 1.3
paper_n_coh = 1.1

## Loading both files

In [ ]:
def first(arr):
    return np.array([a[0] for a in arr])

def load_coh(fname):
    u = uproot.open(os.path.join(other_files_location, fname))
    vt = u["singlephotonana/vertex_tree"]
    a = vt.arrays(["run_number", "subrun_number", "event_number",
                   "reco_vertex_size", "reco_asso_showers", "reco_asso_tracks",
                   "mctruth_daughters_pdg", "mctruth_daughters_status_code", "mctruth_daughters_E",
                   "mctruth_daughters_px", "mctruth_daughters_py", "mctruth_daughters_pz",
                   "mctruth_daughters_startx", "mctruth_daughters_starty", "mctruth_daughters_startz"],
                  library="np")
    pdg = a["mctruth_daughters_pdg"]
    status = a["mctruth_daughters_status_code"]
    assert all(p[0] == 22 for p in pdg), "first daughter is not a photon"
    assert all(abs(p[1]) == 14 and s[1] == 0 for p, s in zip(pdg, status)), "second daughter is not the incoming numu"
    E = first(a["mctruth_daughters_E"]) * 1000
    out = pl.DataFrame({
        "run": a["run_number"], "subrun": a["subrun_number"], "event": a["event_number"],
        "reco_vertex_size": a["reco_vertex_size"],
        "reco_asso_showers": a["reco_asso_showers"], "reco_asso_tracks": a["reco_asso_tracks"],
        "E_gamma": E,
        "costheta": first(a["mctruth_daughters_pz"]) * 1000 / E,
        "px": first(a["mctruth_daughters_px"]),
        "E_nu": np.array([e[1] for e in a["mctruth_daughters_E"]]) * 1000,
        # the photon starts at the neutrino vertex (mctruth_nu_vertex_* is unfilled, -9999)
        "x": first(a["mctruth_daughters_startx"]),
        "y": first(a["mctruth_daughters_starty"]),
        "z": first(a["mctruth_daughters_startz"]),
    })
    pot_tree = u["singlephotonana/pot_tree"].arrays(library="np")
    rs_tree = u["singlephotonana/run_subrun_tree"].arrays(library="np")
    return out, pot_tree, rs_tree

ts, ts_pot, ts_rs = load_coh(TS_FILE)
bb, bb_pot, bb_rs = load_coh(BB_FILE)
print(f"TestSplit: {ts.height:,} events, BatchB: {bb.height:,} events")

## 1. Bookkeeping trees

Every POT branch is zero in both files (`pot_tree.POT`, `run_subrun_tree.subrun_pot`, `pot_per_event`), and `mctruth_nu_vertex_*` / `mctruth_nu_E` are unfilled. So the POT has to come from outside the files.

The event-count branches are still informative:
* `pot_tree.number_of_events`: events processed per job/file.
* `run_subrun_tree.subrun_counts`: events processed per (run, subrun). Run/subrun numbers come from the overlaid data, so a 50-event generation job can span several subruns.

In [ ]:
def summarize(name, df, pot, rs):
    rs_keys = pl.DataFrame({"run": rs["run"], "subrun": rs["subrun"]})
    return {
        "file": name,
        "vertex_tree events": df.height,
        "pot_tree entries": len(pot["POT"]),
        "pot_tree sum(number_of_events)": int(pot["number_of_events"].sum()),
        "sum(POT)": float(pot["POT"].sum()),
        "run_subrun_tree entries": len(rs["run"]),
        "unique (run,subrun) in run_subrun_tree": rs_keys.unique().height,
        "run_subrun_tree sum(subrun_counts)": int(rs["subrun_counts"].sum()),
        "sum(subrun_pot)": float(rs["subrun_pot"].sum()),
        "unique (run,subrun) with events": df.select("run", "subrun").unique().height,
    }

pl.DataFrame([summarize("TestSplit", ts, ts_pot, ts_rs), summarize("BatchB", bb, bb_pot, bb_rs)]).transpose(include_header=True, column_names="file")

For BatchB, `sum(subrun_counts) = pot_tree sum = 67,771 = vertex_tree events`. All generated events are present: an unfiltered sample.

For TestSplit the three numbers differ (132,888 processed in the pot_tree jobs, 34,011 in the run_subrun_tree, 16,139 in the vertex_tree). Next we check whether the two files overlap.

In [ ]:
rse_overlap = ts.join(bb, on=["run", "subrun", "event"], how="inner").height
rs_overlap = ts.select("run", "subrun").unique().join(bb.select("run", "subrun").unique(), on=["run", "subrun"], how="inner").height
# truth-kinematics match (robust to renumbering)
kin = lambda d: d.select(pl.col("E_gamma").round(4), pl.col("px").round(7), pl.col("x").round(3))
kin_overlap = kin(ts).join(kin(bb), on=["E_gamma", "px", "x"], how="inner").height
print(f"RSE overlap: {rse_overlap}, (run,subrun) overlap: {rs_overlap}, identical-photon-kinematics overlap: {kin_overlap}")

The two files are **disjoint**: different generated events.

## 2. TestSplit is a reco-topology-preselected subset

Every TestSplit event has exactly one reconstructed vertex, 1–2 showers and 0–1 tracks. This matches the `preTopo` in its name. BatchB has the full range.

In [ ]:
def topo_table(df, name):
    return (df.group_by("reco_vertex_size", "reco_asso_showers", "reco_asso_tracks").len()
              .with_columns((pl.col("len") / df.height).alias(f"frac_{name}"))
              .rename({"len": f"n_{name}"}))

print("TestSplit topologies:")
print(topo_table(ts, "TS").sort(f"n_TS", descending=True))

bb = bb.with_columns(
    ((pl.col("reco_vertex_size") == 1)
     & pl.col("reco_asso_showers").is_in([1, 2])
     & pl.col("reco_asso_tracks").is_in([0, 1])).alias("passes_preTopo"),
    ((pl.col("reco_vertex_size") == 1) & (pl.col("reco_asso_showers") == 1) & (pl.col("reco_asso_tracks") == 0)).alias("is_1shw0trk"),
)
bb_topo = bb.filter(pl.col("passes_preTopo"))
eff_preTopo_bb = bb_topo.height / bb.height
print(f"\nBatchB fraction passing the same topology (1 vertex, 1-2 showers, 0-1 tracks): {eff_preTopo_bb:.4f}")

cmp = topo_table(ts, "TS").join(topo_table(bb_topo, "BBtopo"), on=["reco_vertex_size", "reco_asso_showers", "reco_asso_tracks"], how="full", coalesce=True)
cmp.sort("n_TS", descending=True)

Within the preselection, the shower/track mix of BatchB matches TestSplit to within ~0.3%, so the two files come from the same generator and reconstruction.

**Number of generated events behind TestSplit.** If the 34,011 in the TestSplit `run_subrun_tree` are the generated events in the kept ("test") subruns, then the preselection efficiency is 16,139 / 34,011. That should equal BatchB's efficiency.

In [ ]:
n_gen_ts = int(ts_rs["subrun_counts"].sum())
eff_ts = ts.height / n_gen_ts
print(f"TestSplit: {ts.height} / {n_gen_ts} = {eff_ts:.4f}   vs BatchB preTopo efficiency {eff_preTopo_bb:.4f}  "
      f"(binomial sigma on the TS ratio {np.sqrt(eff_ts*(1-eff_ts)/n_gen_ts):.4f})")

# per-subrun check: events kept per run_subrun_tree subrun vs subrun_counts
rs_df = pl.DataFrame({"run": ts_rs["run"], "subrun": ts_rs["subrun"], "subrun_counts": ts_rs["subrun_counts"]})
rs_df = rs_df.group_by("run", "subrun").agg(pl.col("subrun_counts").sum())
per_sr = rs_df.join(ts.group_by("run", "subrun").len(), on=["run", "subrun"], how="left").with_columns(pl.col("len").fill_null(0))
not_in_rs = ts.select("run", "subrun").unique().join(rs_df, on=["run", "subrun"], how="anti").height
print(f"TestSplit (run,subrun) with events but absent from run_subrun_tree: {not_in_rs}")
print(f"subruns with more kept events than subrun_counts: {(per_sr['len'] > per_sr['subrun_counts']).sum()}")
n_empty_obs = int((per_sr["len"] == 0).sum())
n_empty_exp = float(((1 - eff_preTopo_bb) ** per_sr["subrun_counts"].to_numpy()).sum())
print(f"run_subrun_tree subruns with zero kept events: observed {n_empty_obs}, expected {n_empty_exp:.1f} for a binomial with p = {eff_preTopo_bb:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
c = per_sr["subrun_counts"].to_numpy(); k = per_sr["len"].to_numpy()
ax.scatter(c + np.random.default_rng(1).uniform(-0.3, 0.3, len(c)), k, s=3, alpha=0.4)
xx = np.arange(0, 51); ax.plot(xx, eff_preTopo_bb * xx, "r-", label=f"BatchB preTopo eff. {eff_preTopo_bb:.3f}")
ax.set_xlabel("run_subrun_tree subrun_counts"); ax.set_ylabel("TestSplit events in that subrun"); ax.legend()
plt.show()

`16,139 / 34,011 = 0.4745`, consistent with BatchB's 0.4736 within binomial error. The number of empty subruns also matches the binomial expectation.

So **TestSplit = all preTopo-passing events out of 34,011 generated events**: the subruns assigned to the test split. The larger `pot_tree` count (132,888) is all events processed by the jobs, including the training subruns. The training/test split is by subrun, so the `run_subrun_tree` keeps only the test subruns, but `pot_tree` is per file.

## 3. Truth distributions: TestSplit is shape-biased, BatchB is not

Compare TestSplit, all of BatchB (the unbiased generated distribution) and BatchB after the same topology cut.

In [ ]:
tpc = dict(xlo=-1.55, xhi=254.8, ylo=-115.53, yhi=117.47, zlo=0.1, zhi=1036.9)  # TPC active volume
def in_fv(d):
    return ((10 < d["x"]) & (d["x"] < 246.4) & (d["y"].abs() < 101.5) & (10 < d["z"]) & (d["z"] < 986.8))

print("vertex ranges (BatchB): x [%.1f, %.1f], y [%.1f, %.1f], z [%.1f, %.1f]" % (
    bb["x"].min(), bb["x"].max(), bb["y"].min(), bb["y"].max(), bb["z"].min(), bb["z"].max()))
tpc_vol = (tpc["xhi"]-tpc["xlo"]) * (tpc["yhi"]-tpc["ylo"]) * (tpc["zhi"]-tpc["zlo"])
fv_vol = 236.4 * 203.0 * 976.8
print(f"FV/TPC volume ratio (expected FV fraction for a uniform-in-TPC generation): {fv_vol/tpc_vol:.4f}")

rows = []
for name, d in [("TestSplit", ts), ("BatchB all", bb), ("BatchB preTopo", bb_topo)]:
    rows.append({"sample": name, "n": d.height, "FV fraction": float(in_fv(d).mean()),
                 "mean costheta": d["costheta"].mean(), "median E_gamma [MeV]": d["E_gamma"].median(),
                 "mean E_gamma [MeV]": d["E_gamma"].mean(), "mean E_nu [MeV]": d["E_nu"].mean()})
pl.DataFrame(rows)

In [ ]:
specs = [("E_gamma", np.linspace(0, 1500, 61), r"true $E_\gamma$ [MeV]"),
         ("costheta", np.linspace(0.0, 1, 51), r"true cos$\theta_\gamma$"),
         ("E_nu", np.linspace(0, 3000, 61), r"true $E_\nu$ [MeV]"),
         ("x", np.linspace(-10, 265, 56), "true vertex x [cm]"),
         ("y", np.linspace(-125, 125, 51), "true vertex y [cm]"),
         ("z", np.linspace(-10, 1050, 54), "true vertex z [cm]")]
fig, axes = plt.subplots(2, 6, figsize=(26, 7), gridspec_kw={"height_ratios": [3, 1]}, sharex="col")
for j, (var, bins, lab) in enumerate(specs):
    hs = {}
    for name, d, col in [("BatchB all", bb, "k"), ("BatchB preTopo", bb_topo, "tab:blue"), ("TestSplit", ts, "tab:red")]:
        h, _ = np.histogram(d[var].to_numpy(), bins=bins)
        hs[name] = (h / h.sum(), np.sqrt(h) / h.sum())
        axes[0, j].stairs(hs[name][0], bins, color=col, label=f"{name} ({d.height:,})")
    axes[0, j].set_ylabel("area normalized")
    for name, col in [("TestSplit", "tab:red"), ("BatchB preTopo", "tab:blue")]:
        with np.errstate(divide="ignore", invalid="ignore"):
            r = hs[name][0] / hs["BatchB all"][0]
            re = r * np.sqrt((hs[name][1] / hs[name][0])**2 + (hs["BatchB all"][1] / hs["BatchB all"][0])**2)
        axes[1, j].errorbar(0.5*(bins[1:]+bins[:-1]), r, yerr=re, fmt=".", color=col, ms=3)
    axes[1, j].axhline(1, color="k", lw=0.8); axes[1, j].set_ylim(0, 2); axes[1, j].set_xlabel(lab)
    axes[1, j].set_ylabel("/ BatchB all")
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.show()

TestSplit (red) sits on BatchB-preTopo (blue) everywhere, and both differ from the full generated sample (black). The preselection removes events near the TPC walls, which raises the FV fraction (0.85 vs 0.76), and it depends on Eγ and cosθ.

The second effect matters for us. The iso1g reweighting uses the TestSplit (cosθ, Eγ) histogram as the coherent target, so the coherent-1γ events we build already carry a Pandora/gLEE preselection efficiency. Our selections then apply their own efficiency on top of that.

The 2D view of the bias in the reweighting binning:

In [ ]:
bins_costheta = np.linspace(-1, 1, 101)
bins_energy = np.concatenate([np.linspace(0, 2000, 101), [1e6]])
H_ts, _, _ = np.histogram2d(ts["costheta"], ts["E_gamma"], bins=[bins_costheta, bins_energy])
H_bb, _, _ = np.histogram2d(bb["costheta"], bb["E_gamma"], bins=[bins_costheta, bins_energy])

coarse = [np.linspace(0.5, 1, 11), np.linspace(0, 1500, 16)]
Hc_ts, _, _ = np.histogram2d(ts["costheta"], ts["E_gamma"], bins=coarse)
Hc_bb, _, _ = np.histogram2d(bb["costheta"], bb["E_gamma"], bins=coarse)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
cmap = mpl.colormaps.get_cmap("viridis").copy(); cmap.set_under("white")
for ax, H, t in [(axes[0], Hc_bb / Hc_bb.sum(), "BatchB (all generated)"), (axes[1], Hc_ts / Hc_ts.sum(), "TestSplit (preTopo)")]:
    m = ax.pcolormesh(coarse[0], coarse[1], H.T, cmap=cmap, vmin=1e-6); plt.colorbar(m, ax=ax); ax.set_title(t)
with np.errstate(divide="ignore", invalid="ignore"):
    R = (Hc_ts / Hc_ts.sum()) / (Hc_bb / Hc_bb.sum())
m = axes[2].pcolormesh(coarse[0], coarse[1], np.where(Hc_bb > 50, R, np.nan).T, cmap="RdBu_r", vmin=0, vmax=2)
plt.colorbar(m, ax=axes[2]); axes[2].set_title("TestSplit / BatchB (area normalized, bins with >50 BatchB events)")
for ax in axes: ax.set_xlabel(r"cos$\theta_\gamma$"); ax.set_ylabel(r"$E_\gamma$ [MeV]")
plt.tight_layout(); plt.show()

## 4. What does 2.22242e24 POT correspond to?

The paper's 28.1% topological efficiency is relative to *all simulated signal events in the TPC*. So `1.1 / (0.281 × 0.3909) = 10.0` is the number of NC coherent events **in the TPC** (not the FV) per 6.87e20 POT. The current code compares it to an FV sum.

First check that the paper's topological requirement (one shower, no tracks) matches the BatchB `1 vertex, 1 shower, 0 tracks` fraction:

In [ ]:
eff_1shw0trk_bb = bb["is_1shw0trk"].mean()
print(f"BatchB 1 vertex / 1 shower / 0 track fraction of all generated (in TPC): {eff_1shw0trk_bb:.4f}   paper topological: {paper_eff_topo}")

paper_tpc_from_coh = paper_n_coh / (paper_eff_topo * paper_eff_rel_coh)
paper_tpc_from_1g = paper_n_1g / (paper_eff_topo * paper_eff_rel_1g)
# rounding: the quoted 1.1 / 1.3 are known to +-0.05
print(f"paper-implied NC coherent events in the TPC per 6.87e20 POT: "
      f"{paper_tpc_from_coh:.2f} [{(paper_n_coh-0.05)/(paper_eff_topo*paper_eff_rel_coh):.2f}, {(paper_n_coh+0.05)/(paper_eff_topo*paper_eff_rel_coh):.2f}] from the coherent-rich 1.1, "
      f"{paper_tpc_from_1g:.2f} [{(paper_n_1g-0.05)/(paper_eff_topo*paper_eff_rel_1g):.2f}, {(paper_n_1g+0.05)/(paper_eff_topo*paper_eff_rel_1g):.2f}] from the single-photon 1.3")

BatchB's 1-shower-0-track fraction reproduces the paper's 28.1%, which confirms the "all simulated in the TPC" denominator and again shows BatchB is the unbiased generated sample.

Now try each candidate event count as "the events generated in 2.22242e24 POT":

In [ ]:
cands = {
    "16,139 (TestSplit vertex_tree; current code)": ts.height,
    "34,011 (TestSplit run_subrun_tree = generated behind TestSplit)": n_gen_ts,
    "67,771 (BatchB, all generated)": bb.height,
    "132,888 (TestSplit pot_tree, all jobs incl. training subruns)": int(ts_pot["number_of_events"].sum()),
}
rows = []
for name, n in cands.items():
    n_tpc = n / MARK_POT * PAPER_POT
    rows.append({"N generated in 2.22242e24 POT": name,
                 "in-TPC events / 6.87e20": n_tpc,
                 "pred. coherent-rich sel. (paper 1.1)": n_tpc * paper_eff_topo * paper_eff_rel_coh,
                 "pred. single-photon sel. (paper 1.3)": n_tpc * paper_eff_topo * paper_eff_rel_1g,
                 "ratio to paper (coh-rich)": n_tpc / paper_tpc_from_coh})
pl.Config.set_tbl_width_chars(250); pl.Config.set_fmt_str_lengths(80)
pl.DataFrame(rows)

**34,011 generated events ↔ 2.22242e24 POT reproduces the paper.** It predicts 1.16 coherent-rich events (paper 1.1) and 1.36 single-photon events (paper 1.3), about 4–5% high. That is within the rounding of the quoted numbers plus small differences between our efficiencies and the paper's. The other candidates miss by factors of 0.5, 2.0 and 3.9.

The natural reading is that Mark's POT is the POT of the test-split subruns behind the TestSplit file. Normalizing that file by its *vertex_tree* count (16,139) in the current code is the mistake: 16,139 is the preselected count, not the generated count.

The same rate gives a POT for BatchB:

In [ ]:
rate_per_pot_tpc = n_gen_ts / MARK_POT
bb_pot_equiv = bb.height / rate_per_pot_tpc
print(f"NC coherent 1g rate in the TPC: {rate_per_pot_tpc:.4e} events/POT  ({rate_per_pot_tpc*PAPER_POT:.2f} per 6.87e20)")
print(f"equivalent BatchB POT: {bb_pot_equiv:.4e}")
print(f"  -> FV rate {rate_per_pot_tpc*PAPER_POT*float(in_fv(bb).mean()):.2f} per 6.87e20 (FV fraction {float(in_fv(bb).mean()):.3f})")

BatchB was a separate generation batch (no shared events or subruns), but from the same generator, flux and reconstruction. So it can be normalized with the same rate: about 4.43e24 POT-equivalent. No separate POT number is needed for it.

## 5. Consequences for the iso1g reweighting

The reweighting maps iso1g events to the coherent (cosθ, Eγ) distribution, then normalizes. Load the iso1g events and the current saved weights.

In [ ]:
iso = (pl.scan_parquet(f"{intermediate_files_location}/all_df.parquet")
       .filter(pl.col("filetype") == "isotropic_one_gamma_overlay")
       .select("run", "subrun", "event", "wc_true_leading_shower_energy", "wc_true_leading_shower_costheta",
               "wc_net_weight_open_data", "wc_truth_vtxX", "wc_truth_vtxY", "wc_truth_vtxZ")
       .collect()
       .rename({"wc_true_leading_shower_energy": "E_gamma", "wc_true_leading_shower_costheta": "costheta",
                "wc_truth_vtxX": "x", "wc_truth_vtxY": "y", "wc_truth_vtxZ": "z"}))
iso = iso.with_columns(
    ((tpc["xlo"] < pl.col("x")) & (pl.col("x") < tpc["xhi"]) & (tpc["ylo"] < pl.col("y")) & (pl.col("y") < tpc["yhi"])
     & (tpc["zlo"] < pl.col("z")) & (pl.col("z") < tpc["zhi"])).alias("in_tpc"),
    ((10 < pl.col("x")) & (pl.col("x") < 246.4) & (pl.col("y").abs() < 101.5) & (10 < pl.col("z")) & (pl.col("z") < 986.8)).alias("in_fv"),
)
print(f"iso1g events: {iso.height:,}; in TPC: {iso['in_tpc'].mean():.3f}; FV fraction of in-TPC: {iso.filter('in_tpc')['in_fv'].mean():.3f} "
      f"(uniform expectation {fv_vol/tpc_vol:.3f})")
w = iso["wc_net_weight_open_data"].to_numpy()
print(f"mean iso1g open-data weight: {w.mean():.4f}  (1/mean = {1/w.mean():.2f})")

fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for ax, v, b in zip(axes, "xyz", [np.linspace(-200, 450, 66), np.linspace(-300, 300, 61), np.linspace(-300, 1350, 67)]):
    ax.hist(iso[v], bins=b, histtype="step", label="iso1g"); ax.set_xlabel(f"true vertex {v} [cm]")
    ax.axvline(tpc[v+"lo"], color="r", ls="--"); ax.axvline(tpc[v+"hi"], color="r", ls="--", label="TPC")
axes[0].legend(); plt.tight_layout(); plt.show()

In [ ]:
cur = pl.read_parquet(f"{intermediate_files_location}/coh_1g_reweighting.parquet")
isoc = iso.join(cur.select("run", "subrun", "event", "coherent_1g_weight_per_pot"), on=["run", "subrun", "event"], how="left") \
          .with_columns(pl.col("coherent_1g_weight_per_pot").fill_null(0.0))
cur_tot = isoc["coherent_1g_weight_per_pot"].sum() * PAPER_POT
cur_tpc = isoc.filter("in_tpc")["coherent_1g_weight_per_pot"].sum() * PAPER_POT
cur_fv = isoc.filter("in_fv")["coherent_1g_weight_per_pot"].sum() * PAPER_POT
print(f"CURRENT weights, events per 6.87e20: all iso1g {cur_tot:.2f}, in TPC {cur_tpc:.2f}, in FV {cur_fv:.2f}")
print(f"  target from section 4: in TPC {rate_per_pot_tpc*PAPER_POT:.2f}, in FV {rate_per_pot_tpc*PAPER_POT*float(in_fv(bb).mean()):.2f}")

The current weights are off in several partly cancelling ways:
* `H_iso1g` is filled with the open-data weights but the weight is not reapplied per event, giving a factor ≈ 1/⟨w⟩.
* The reference count is the preselected 16,139 instead of the generated 34,011.
* There is the 0.5 "fudge".
* The target shape is the preselected TestSplit.
* About half of the iso1g events lie outside the TPC but still get weights.

### Proposed normalization

Build the ratio from iso1g events **inside the TPC**, unweighted, against **BatchB** (unbiased, 4× the statistics). Normalize with BatchB's POT-equivalent:

`w_per_pot(bin) = H_BatchB(bin) / H_iso1g,inTPC(bin) / POT_BatchB`, with `POT_BatchB = 67,771 × 2.22242e24 / 34,011`.

Iso1g events inside the TPC then sum exactly to the coherent rate in the TPC. Events outside the TPC get the same per-event weight; whether that is right is checked below.

In [ ]:
# iso1g vertices fill a cylinder (the cryostat) around the TPC: check the density per unit volume inside vs outside the TPC
xc, yc = 0.5 * (iso["x"].min() + iso["x"].max()), 0.5 * (iso["y"].min() + iso["y"].max())
r = np.hypot(iso["x"].to_numpy() - xc, iso["y"].to_numpy() - yc)
R = r.max(); zmin, zmax = iso["z"].min(), iso["z"].max()
cyl_vol = np.pi * R**2 * (zmax - zmin)
n_in = int(iso["in_tpc"].sum()); n_out = iso.height - n_in
print(f"iso1g cylinder: axis (x={xc:.1f}, y={yc:.1f}), R = {R:.1f} cm, z [{zmin:.0f}, {zmax:.0f}]")
print(f"density in TPC {n_in/tpc_vol:.3e} /cm3, outside TPC (within the cylinder) {n_out/(cyl_vol-tpc_vol):.3e} /cm3")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist((r / R)**2, bins=20, range=(0, 1), histtype="step"); axes[0].set_xlabel(r"$(r/R)^2$ (flat = uniform in the disk)")
axes[1].hist(iso["z"], bins=40, histtype="step"); axes[1].set_xlabel("true vertex z [cm]")
plt.tight_layout(); plt.show()

The iso1g vertices are spread roughly uniformly through the cryostat cylinder, with about the same density per unit volume inside and outside the TPC. So the per-event coherent weight derived from in-TPC events gives a physically sensible coherent rate outside the TPC too (NC coherent on the cryostat argon; those events reach our selections only rarely). The alternative is zero weight outside the TPC, since the reference generation only covers the TPC. Either way, **the normalization must be defined on in-TPC iso1g events only**.

In [ ]:
iso_tpc = iso.filter("in_tpc")
H_iso_tpc, _, _ = np.histogram2d(iso_tpc["costheta"], iso_tpc["E_gamma"], bins=[bins_costheta, bins_energy])

def per_pot_weights(H_ref, ref_pot):
    ratio = np.where(H_iso_tpc > 0, H_ref / np.where(H_iso_tpc > 0, H_iso_tpc, 1), 0.0) / ref_pot
    i_c = np.clip(np.digitize(iso["costheta"], bins_costheta) - 1, 0, len(bins_costheta) - 2)
    i_e = np.clip(np.digitize(iso["E_gamma"], bins_energy) - 1, 0, len(bins_energy) - 2)
    lost = H_ref[(H_iso_tpc == 0)].sum() / H_ref.sum()
    return ratio[i_c, i_e], lost

ts_equiv_pot = MARK_POT * ts.height / n_gen_ts   # TestSplit treated as a preTopo-biased sample of 2.22e24 POT
rows = []
for name, H_ref, pot in [("BatchB, POT = 67771/34011 x 2.22242e24", H_bb, bb_pot_equiv),
                         ("TestSplit, POT = 2.22242e24 (preTopo-biased shape)", H_ts, MARK_POT)]:
    wpp, lost = per_pot_weights(H_ref, pot)
    rows.append({"reference": name,
                 "in TPC / 6.87e20": wpp[iso["in_tpc"].to_numpy()].sum() * PAPER_POT,
                 "in FV / 6.87e20": wpp[iso["in_fv"].to_numpy()].sum() * PAPER_POT,
                 "all iso1g / 6.87e20": wpp.sum() * PAPER_POT,
                 "ref. fraction in bins w/o iso1g": lost,
                 "n iso1g w>0": int((wpp > 0).sum())})
rows.append({"reference": "CURRENT saved weights", "in TPC / 6.87e20": cur_tpc, "in FV / 6.87e20": cur_fv,
             "all iso1g / 6.87e20": cur_tot, "ref. fraction in bins w/o iso1g": None, "n iso1g w>0": int((isoc["coherent_1g_weight_per_pot"] > 0).sum())})
pl.DataFrame(rows)

For comparison: the paper-implied in-TPC number is ≈ 10.0 (coherent-rich) / 10.1 (single-photon) per 6.87e20, and Mark's POT with 34,011 gives 10.5.

## Summary

1. **BatchB** (Mark's Jan2022 gLEE files) is a complete, unfiltered generation: 67,771 events, uniform in the TPC (FV fraction 0.76 = the FV/TPC volume ratio). Its 1-shower-0-track fraction reproduces the paper's 28.1% topological efficiency.
2. **TestSplit** (gge's flux-assignment file) is disjoint from BatchB but uses the same generator and reconstruction. It is the **preTopo-selected** subset (1 vertex, 1–2 showers, 0–1 tracks; 47.4% efficiency) of the **34,011** events generated in its test-split subruns (`run_subrun_tree.subrun_counts`). Its (cosθ, Eγ, vertex) distributions carry that preselection bias.
3. **2.22242e24 POT ↔ 34,011 generated events** reproduces the paper's predicted selected events to about 5%. The other possible counts (16,139 / 67,771 / 132,888) miss by factors of 0.5 to 3.9. Applying the same rate, BatchB corresponds to ≈ 4.43e24 POT.
4. The paper's `1.1 / (0.281 × 0.3909) ≈ 10` is an **in-TPC** count, not FV. The check in `coh_1g_reweighting.py` compares it to an FV sum.
5. The current weights land near the paper only through cancelling errors: the weighted `H_iso1g`, 16,139 instead of 34,011, the 0.5 fudge, and the biased shape. They give 12.7 in the TPC per 6.87e20 against the target 10.5, plus 16 more events from out-of-TPC iso1g.
6. Only 44% of the iso1g vertices are in the TPC. The rest fill the cryostat at the same density, so the normalization has to be defined using in-TPC iso1g events.

**Recommendation:**
* Use BatchB as the shape reference, plus its true E_ν for the flux-donor assignment.
* Build `H_iso1g` from unweighted in-TPC iso1g events.
* Normalize with POT_BatchB = 67,771 × 2.22242e24 / 34,011.
* Drop the fudge factor.
* Validate against the paper's in-TPC number (≈10) instead of FV.

Anchoring directly to the paper number would give the same result to about 5%, so the POT-based normalization is consistent and preferable.